# Taller: Detección de Bordes y Contornos
**Asignatura:** Computación Visual

Este Jupyter Notebook contiene el desarrollo paso a paso del taller de detección de bordes y contornos (Puntos 1 a 5 y parte del 7 de la guía).

## Objetivos del Taller:
1. **Operadores de bordes**: Sobel, Prewitt, Laplaciano y Scharr en escala de grises.
2. **Detector de Canny**: Experimentar con umbrales y suavizado Gaussiano previo (analizando el efecto de sigma).
3. **Detección de Contornos**: Binarización adaptativa, jerarquías de contornos y filtrado de áreas.
4. **Aproximación de Formas**: Reducción de vértices (`approxPolyDP`), cálculo de área y perímetro, y clasificación geométrica.
5. **Análisis de Momentos**: Cálculo de centroides, orientación y excentricidad geométrica.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import filters
import os

# Asegurarnos de que las figuras se rendericen en el notebook
%matplotlib inline

# Rutas a las imágenes de prueba
media_dir = os.path.join("..", "media")
natural_path = os.path.join(media_dir, "natural.png")
shapes_path = os.path.join(media_dir, "shapes.png")

print("Librerías importadas correctamente!")

## 1. Operadores Básicos de Detección de Bordes

Aplicaremos los operadores de gradiente de primer orden (Sobel, Prewitt, Scharr) y segundo orden (Laplaciano) sobre nuestra imagen de textura compleja (`natural.png`).

In [ ]:
# 1. Cargar imagen en escala de grises
img_natural = cv2.imread(natural_path, cv2.IMREAD_GRAYSCALE)

# 2. Operador Sobel (X, Y y Magnitud)
sobelx = cv2.Sobel(img_natural, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(img_natural, cv2.CV_64F, 0, 1, ksize=3)
sobel_mag = np.sqrt(sobelx**2 + sobely**2)

# 3. Operador Scharr (Variante mejorada de Sobel, ksize=-1 activa Scharr de forma interna)
scharrx = cv2.Scharr(img_natural, cv2.CV_64F, 1, 0)
scharry = cv2.Scharr(img_natural, cv2.CV_64F, 0, 1)
scharr_mag = np.sqrt(scharrx**2 + scharry**2)

# 4. Operador Prewitt (Usando scikit-image para exactitud)
# filters.prewitt de skimage trabaja con floats en rango [0, 1]
img_normalized = img_natural.astype(np.float64) / 255.0
prewitt_mag = filters.prewitt(img_normalized)

# 5. Operador Laplaciano (Gradiente de 2do orden)
laplacian = cv2.Laplacian(img_natural, cv2.CV_64F)

# Visualizar resultados
plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
plt.imshow(img_natural, cmap='gray')
plt.title('Original (Gris)')
plt.axis('off')

plt.subplot(2, 3, 2)
plt.imshow(np.abs(sobel_mag), cmap='gray')
plt.title('Sobel Magnitud')
plt.axis('off')

plt.subplot(2, 3, 3)
plt.imshow(np.abs(prewitt_mag), cmap='gray')
plt.title('Prewitt Magnitud')
plt.axis('off')

plt.subplot(2, 3, 4)
plt.imshow(np.abs(laplacian), cmap='gray')
plt.title('Laplaciano (2do Orden)')
plt.axis('off')

plt.subplot(2, 3, 5)
plt.imshow(np.abs(scharr_mag), cmap='gray')
plt.title('Scharr Magnitud')
plt.axis('off')

plt.tight_layout()
# Guardamos la comparación de operadores en media
plt.savefig(os.path.join(media_dir, "1_sobel_prewitt_laplacian_scharr.png"), dpi=300)
plt.show()

### Análisis Comparativo de Operadores
- **Sobel**: Excelente para bordes gruesos y suavizado de ruido implícito por su kernel bilineal.
- **Prewitt**: Kernels simples que son ligeramente más sensibles a ruido de alta frecuencia que Sobel.
- **Scharr**: Proporciona una simetría rotacional mucho más precisa, detectando bordes diagonales con gran definición.
- **Laplaciano**: Al ser de segundo orden, destaca las transiciones más rápidas y es extremadamente sensible a cambios finos de frecuencia (incluyendo ruido), produciendo líneas muy delgadas pero con ruido de sal y pimienta.

## 2. Detector de Bordes de Canny

El detector de Canny es un operador óptimo de varias etapas que reduce los bordes anchos a un ancho de un píxel (adelgazamiento de bordes) y usa umbrales con histéresis.

In [ ]:
# 1. Aplicar Canny con diferentes combinaciones de umbrales
canny_low = cv2.Canny(img_natural, 30, 80)
canny_mid = cv2.Canny(img_natural, 80, 150)
canny_high = cv2.Canny(img_natural, 150, 220)

# 2. Suavizado Gaussiano previo con diferentes sigmas para analizar su efecto
blur_s1 = cv2.GaussianBlur(img_natural, (5, 5), 1.0)
blur_s3 = cv2.GaussianBlur(img_natural, (11, 11), 3.0)

canny_blur_s1 = cv2.Canny(blur_s1, 80, 150)
canny_blur_s3 = cv2.Canny(blur_s3, 80, 150)

# Visualización comparativa
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(img_natural, cmap='gray')
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

axes[0, 1].imshow(canny_low, cmap='gray')
axes[0, 1].set_title('Canny (Low Thresh: 30-80)')
axes[0, 1].axis('off')

axes[0, 2].imshow(canny_mid, cmap='gray')
axes[0, 2].set_title('Canny (Mid Thresh: 80-150)')
axes[0, 2].axis('off')

axes[1, 0].imshow(canny_high, cmap='gray')
axes[1, 0].set_title('Canny (High Thresh: 150-220)')
axes[1, 0].axis('off')

axes[1, 1].imshow(canny_blur_s1, cmap='gray')
axes[1, 1].set_title('Canny (80-150) + Gauss Sigma=1')
axes[1, 1].axis('off')

axes[1, 2].imshow(canny_blur_s3, cmap='gray')
axes[1, 2].set_title('Canny (80-150) + Gauss Sigma=3')
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(media_dir, "2_canny_thresholds_sigma.png"), dpi=300)
plt.show()

### Análisis del Efecto de los Umbrales y el Sigma:
- **Umbrales Bajos**: Al disminuir los umbrales de histéresis, se conectan bordes más débiles, capturando texturas finas pero aumentando el ruido.
- **Umbrales Altos**: Solo se conservan bordes muy marcados (fuertes contrastes), eliminando detalles secundarios de sombreado o ruido.
- **Efecto de Sigma (Suavizado)**: Un mayor valor de `sigma` elimina frecuencias más altas de la imagen antes del cálculo del gradiente. Esto causa que Canny ignore texturas pequeñas y detalles finos, preservando solo las estructuras principales de contorno global.

## 3. Detección y Dibujo de Contornos

Trabajaremos con la imagen `shapes.png` que contiene formas geométricas aisladas para demostrar binarización y el dibujo y filtrado de contornos.

In [ ]:
# 1. Cargar imagen de figuras en escala de grises
img_shapes = cv2.imread(shapes_path)
gray_shapes = cv2.cvtColor(img_shapes, cv2.COLOR_BGR2GRAY)

# 2. Binarización con umbral adaptativo
thresh_shapes = cv2.adaptiveThreshold(gray_shapes, 255, 
                                      cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                      cv2.THRESH_BINARY, 11, 2)

# Invertir la imagen binaria para que las formas sean blancas (255) y el fondo negro (0)
thresh_shapes_inv = cv2.bitwise_not(thresh_shapes)

# 3. Encontrar contornos con jerarquía árbol completo
# RETR_TREE recupera todos los contornos y reconstruye una jerarquía completa de anidamiento.
contours, hierarchy = cv2.findContours(thresh_shapes_inv, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

# Clonar imagen para dibujar sobre ella
img_contours_all = img_shapes.copy()
img_contours_filtered = img_shapes.copy()

# 4. Dibujar todos los contornos (Azul)
cv2.drawContours(img_contours_all, contours, -1, (255, 0, 0), 3)

# 5. Filtrar contornos por área mínima/máxima (Eliminar texto o pequeños ruidos)
# Por ejemplo, filtrar para conservar solo contornos entre 2000px y 25000px
filtered_contours = []
for cnt in contours:
    area = cv2.contourArea(cnt)
    if 2000 < area < 25000:
        filtered_contours.append(cnt)

# Dibujar contornos filtrados (Verde)
cv2.drawContours(img_contours_filtered, filtered_contours, -1, (0, 255, 0), 3)

# Mostrar binarización y resultados
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(thresh_shapes_inv, cmap='gray')
axes[0].set_title('Binarización Adaptativa Invertida')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img_contours_all, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Todos los Contornos ({len(contours)} detectados)')
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(img_contours_filtered, cv2.COLOR_BGR2RGB))
axes[2].set_title(f'Contornos Filtrados por Área ({len(filtered_contours)} formas)')
axes[2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(media_dir, "3_contours_adaptive_hierarchy.png"), dpi=300)
plt.show()

## 4. Aproximación de Formas y Clasificación Geométrica

Usando el algoritmo Ramer-Douglas-Peucker (`cv2.approxPolyDP()`), reduciremos la cantidad de vértices de los contornos para clasificar las figuras según su número de lados.

In [ ]:
img_shapes_classified = img_shapes.copy()

for idx, cnt in enumerate(filtered_contours):
    # 1. Calcular perímetro (longitud del arco)
    perimeter = cv2.arcLength(cnt, True)
    
    # 2. Calcular área
    area = cv2.contourArea(cnt)
    
    # 3. Aproximar el contorno con un polígono
    # epsilon especifica la distancia máxima de tolerancia entre el contorno original y el aproximado
    epsilon = 0.03 * perimeter
    approx = cv2.approxPolyDP(cnt, epsilon, True)
    vertices = len(approx)
    
    # 4. Clasificación según número de vértices
    shape_name = "Desconocido"
    color = (255, 255, 255)
    
    if vertices == 3:
        shape_name = "Triangulo"
        color = (0, 0, 255)
    elif vertices == 4:
        # Comprobar si es cuadrado o rectángulo usando el aspect ratio de su cuadro delimitador
        x, y, w, h = cv2.boundingRect(cnt)
        aspect_ratio = float(w)/h
        shape_name = "Cuadrado" if 0.95 <= aspect_ratio <= 1.05 else "Rectangulo"
        color = (0, 255, 0)
    elif vertices == 5:
        shape_name = "Pentagono"
        color = (255, 0, 255)
    elif vertices == 10:
        shape_name = "Estrella"
        color = (0, 255, 255)
    elif vertices > 5:
        shape_name = "Circulo"
        color = (255, 255, 0)
        
    # Calcular momentos para poner el texto en el centroide
    M = cv2.moments(cnt)
    if M['m00'] != 0:
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
    else:
        cx, cy = 0, 0
        
    # Dibujar polígono aproximado y colocar etiqueta
    cv2.drawContours(img_shapes_classified, [approx], -1, color, 3)
    cv2.putText(img_shapes_classified, f"{shape_name} ({vertices}v)", (cx - 45, cy), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(img_shapes_classified, f"{shape_name} ({vertices}v)", (cx - 45, cy), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
                
    print(f"Forma #{idx+1}: {shape_name} | Vértices original={len(cnt)} -> Aprox={vertices} | Área={area} | Perímetro={round(perimeter,2)}")

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(img_shapes_classified, cv2.COLOR_BGR2RGB))
plt.title('Aproximación Poligonal y Clasificación de Formas')
plt.axis('off')
plt.savefig(os.path.join(media_dir, "4_polygon_approximation_shapes.png"), dpi=300)
plt.show()

## 5. Análisis de Momentos y Características Geométricas

Los momentos de una imagen/contorno (`cv2.moments()`) permiten extraer descriptores matemáticos espaciales invariantes al tamaño y posición como el centroide, la orientación (ángulo de rotación) y la excentricidad (elongación).

In [ ]:
img_moments = img_shapes.copy()

print("\n--- CÁLCULO DE PROPIEDADES MEDIANTE MOMENTOS DE HU ---")
for idx, cnt in enumerate(filtered_contours):
    # 1. Obtener momentos espaciales, centrales y normalizados
    M = cv2.moments(cnt)
    
    # 2. Centroide
    cx = int(M['m10'] / M['m00'])
    cy = int(M['m01'] / M['m00'])
    
    # 3. Orientación (Ángulo theta en radianes usando los momentos centrales de 2do orden)
    # formula: theta = 0.5 * arctan(2 * mu11 / (mu20 - mu02))
    mu11 = M['mu11']
    mu20 = M['mu20']
    mu02 = M['mu02']
    
    diff = mu20 - mu02
    theta = 0.5 * np.arctan2(2 * mu11, diff) if diff != 0 else 0.0
    angle_deg = np.rad2deg(theta)
    
    # 4. Excentricidad (Elongación de la forma, 0 para círculos, cercano a 1 para líneas)
    # formula: e = sqrt(1 - (L_min / L_maj)^2)
    term1 = mu20 + mu02
    term2 = np.sqrt(4 * (mu11**2) + (mu20 - mu02)**2)
    l1 = (term1 + term2) / 2.0  # Semieje mayor
    l2 = (term1 - term2) / 2.0  # Semieje menor
    eccentricity = np.sqrt(1 - (l2 / l1)) if l1 > 0 and l2 >= 0 else 0.0
    
    # 5. Obtener los 7 Momentos de Hu (invariantes a escala, traslación y rotación)
    hu_moments = cv2.HuMoments(M).flatten()
    # Escala logarítmica para visualización más entendible de Hu
    hu_log = -np.sign(hu_moments) * np.log10(np.abs(hu_moments) + 1e-10)
    
    # Graficar ejes de orientación sobre la imagen
    # Dibujamos una línea en la dirección de la orientación
    line_length = 50
    xe = int(cx + line_length * np.cos(theta))
    ye = int(cy + line_length * np.sin(theta))
    
    # Dibujar centroide (Rojo)
    cv2.circle(img_moments, (cx, cy), 6, (0, 0, 255), -1)
    # Dibujar eje de orientación (Blanco)
    cv2.line(img_moments, (cx, cy), (xe, ye), (255, 255, 255), 3)
    # Dibujar texto de excentricidad
    cv2.putText(img_moments, f"Ecc:{round(eccentricity, 2)}", (cx - 30, cy + 25), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(img_moments, f"Ecc:{round(eccentricity, 2)}", (cx - 30, cy + 25), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)
                
    print(f"Figura #{idx+1}: Centroide=({cx}, {cy}) | Orientación={round(angle_deg,1)}° | Excentricidad={round(eccentricity,3)}")
    print(f"   Hu log: {np.round(hu_log[:3], 3)} (Primeros 3 de 7)")

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(img_moments, cv2.COLOR_BGR2RGB))
plt.title('Centroides (Rojo), Ejes de Orientación (Blanco) y Excentricidad')
plt.axis('off')
plt.savefig(os.path.join(media_dir, "5_moments_centroids_orientation.png"), dpi=300)
plt.show()